# Observabilidade e Avaliação

A resposta final de um agente esconde o caminho até ela: quais ferramentas foram chamadas, com quais argumentos e quantos tokens cada etapa gastou. Este notebook inspeciona o trace de um agente de livraria, avalia o agente com um dataset de casos e um LLM-as-judge e usa a mesma avaliação para comparar versões pelo acerto e pelo custo.

In [1]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langfuse

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
# os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")

In [2]:
import time

import pandas as pd
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth", None)

## Tracing

O atendente da Livraria Sebo Azul tem duas ferramentas: a busca no catálogo, que devolve id, título, autor e preço, e a consulta de estoque pelo id do livro.

In [3]:
BOOKS = {
    "b1": {"title": "Duna", "author": "Frank Herbert", "price": 89.9, "stock": 4},
    "b2": {"title": "Fundação", "author": "Isaac Asimov", "price": 59.9, "stock": 0},
    "b3": {"title": "Neuromancer", "author": "William Gibson", "price": 64.9, "stock": 2},
    "b4": {"title": "O Hobbit", "author": "J. R. R. Tolkien", "price": 49.9, "stock": 7},
}

In [4]:
@tool
def search_books(query: str) -> list[dict]:
    """Busca livros no catálogo por título ou autor e devolve id, título, autor e preço."""
    words = set(query.lower().split())
    found = []
    for id, book in BOOKS.items():
        if words & set(f"{book['title']} {book['author']}".lower().split()):
            found.append({"id": id, "title": book["title"], "author": book["author"], "price": book["price"]})
    return found


@tool
def get_stock(book_id: str) -> int:
    """Devolve a quantidade em estoque de um livro pelo id."""
    return BOOKS[book_id]["stock"]

In [5]:
SYSTEM = """Você é o atendente da Livraria Sebo Azul.
Responda em uma frase."""

agent = create_agent(
    init_chat_model("openai:gpt-4.1-mini", temperature=0.0),
    tools=[search_books, get_stock],
    system_prompt=SYSTEM,
)

In [6]:
result = agent.invoke({"messages": [{"role": "user", "content": "Vocês têm Duna em estoque?"}]})
print(result["messages"][-1].text)

Sim, temos o livro "Duna" de Frank Herbert em estoque, com 4 exemplares disponíveis.


A lista `result["messages"]` guarda o trace completo da execução: cada tool call, com os argumentos, e o resultado que a ferramenta devolveu.

In [7]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Vocês têm Duna em estoque?
================================== Ai Message ==================================
Tool Calls:
  search_books (call_XAYCzb42CmIWesOKM8kdCE1Q)
 Call ID: call_XAYCzb42CmIWesOKM8kdCE1Q
  Args:
    query: Duna
================================= Tool Message =================================
Name: search_books

[{"id": "b1", "title": "Duna", "author": "Frank Herbert", "price": 89.9}]
================================== Ai Message ==================================
Tool Calls:
  get_stock (call_uK2J6NqyyX16boqourqfMfxN)
 Call ID: call_uK2J6NqyyX16boqourqfMfxN
  Args:
    book_id: b1
================================= Tool Message =================================
Name: get_stock

4
================================== Ai Message ==================================

Sim, temos o livro "Duna" de Frank Herbert em estoque, com 4 exemplares disponíveis.


O agente buscou "Duna", recebeu o id `b1` e usou esse id na consulta de estoque antes de responder. Cada mensagem do modelo também registra o consumo de tokens em `usage_metadata`.

In [8]:
for message in result["messages"]:
    if message.type == "ai":
        usage = message.usage_metadata
        print(f"entrada: {usage['input_tokens']}, saída: {usage['output_tokens']}")

entrada: 111, saída: 15
entrada: 164, saída: 16
entrada: 189, saída: 23


Cada chamada ao modelo reenvia a conversa inteira, então os tokens de entrada crescem a cada chamada.

### Langfuse

Em produção, o trace de cada execução é gravado por uma plataforma de observabilidade. O Langfuse, de código aberto, recebe as etapas do agente por um callback no `config` do `invoke` e lê as chaves `LANGFUSE_PUBLIC_KEY` e `LANGFUSE_SECRET_KEY` de um projeto, criado:

- na nuvem, em cloud.langfuse.com;
- ou localmente, com `docker compose up` no repositório do Langfuse e `LANGFUSE_BASE_URL=http://localhost:3000`, fora do Colab.

In [9]:
langfuse = CallbackHandler()

agent.invoke(
    {"messages": [{"role": "user", "content": "Vocês têm Duna em estoque?"}]},
    config={"callbacks": [langfuse]},
)
get_client().flush()
get_client().get_trace_url(trace_id=langfuse.last_trace_id)

'https://cloud.langfuse.com/project/cmw9i3is302klad0dr7hy3pww/traces/8ce1dee31fe42712275cc00915de73b8'

Os eventos saem em lote, e o `flush` força o envio dos pendentes antes de pedir o endereço. O trace aparece como uma árvore de spans, um para cada chamada ao modelo e a cada ferramenta, com os tokens, a latência e o custo de cada um. O LangSmith, da LangChain, faz o mesmo apenas com variáveis de ambiente, sem callback.

### Exercício 1

Pergunte ao agente "Têm Duna e Neuromancer em estoque?", passando o callback do Langfuse, e abra o trace. Quantas tool calls foram feitas, e quantos tokens a execução gastou no total?

In [10]:
# Seu código aqui

## Dataset de avaliação

Avaliar um agente exige um dataset de casos com resultado conhecido. Cada caso traz a pergunta, as ferramentas que deveriam ser chamadas, em ordem, e uma resposta de referência. O dataset inclui um livro fora do catálogo e uma pergunta que nenhuma ferramenta responde.

In [11]:
CASES = [
    {"question": "Vocês têm Duna em estoque?", "tools": ["search_books", "get_stock"], "reference": "Sim, há 4 exemplares."},
    {"question": "Quanto custa Neuromancer?", "tools": ["search_books"], "reference": "Custa R$ 64,90."},
    {"question": "Fundação está disponível?", "tools": ["search_books", "get_stock"], "reference": "Está sem estoque."},
    {"question": "Têm A Revolução dos Bichos?", "tools": ["search_books"], "reference": "Não está no catálogo."},
    {"question": "Qual o horário da loja?", "tools": [], "reference": "O atendente não tem essa informação."},
]

A primeira métrica olha só o trace: as ferramentas chamadas, na ordem em que foram chamadas, são as esperadas pelo caso.

In [12]:
rows = []
for case in CASES:
    messages = agent.invoke({"messages": [{"role": "user", "content": case["question"]}]})["messages"]
    called = []
    for message in messages:
        if message.type == "ai":
            called += [call["name"] for call in message.tool_calls]
    rows.append({"question": case["question"], "tools_ok": called == case["tools"], "answer": messages[-1].text})

pd.DataFrame(rows)

,question,tools_ok,answer
0,Vocês têm Duna em estoque?,True,"Sim, temos o livro ""Duna"" de Frank Herbert em estoque, com 4 unidades disponíveis."
1,Quanto custa Neuromancer?,True,"O livro Neuromancer custa R$64,90."
2,Fundação está disponível?,True,"O livro ""Fundação"" de Isaac Asimov está esgotado no momento."
3,Têm A Revolução dos Bichos?,True,"No momento, não temos ""A Revolução dos Bichos"" disponível em nosso catálogo. Posso ajudar com outro título?"
4,Qual o horário da loja?,True,"A Livraria Sebo Azul funciona de segunda a sexta, das 9h às 18h, e aos sábados, das 9h às 14h."


As ferramentas bateram nos cinco casos, inclusive no do horário, em que nenhuma deveria ser chamada. A resposta desse caso, porém, traz um horário que não existe em lugar nenhum.

## LLM-as-judge

Comparar a resposta com a referência por igualdade de texto falharia em quase todos os casos, porque a mesma informação pode ser escrita de muitas formas. Um LLM-as-judge recebe a pergunta, a referência e a resposta e devolve um veredito com saída estruturada.

In [13]:
class Verdict(BaseModel):
    reason: str = Field(description="justificativa em uma frase")
    correct: bool = Field(description="se a resposta traz a mesma informação da referência, sem inventar nada")


judge = init_chat_model("openai:gpt-4.1-mini", temperature=0.0).with_structured_output(Verdict)

JUDGE = """Avalie a resposta de um atendente de livraria.

Pergunta: {question}
Referência: {reference}
Resposta: {answer}"""

In [14]:
for row, case in zip(rows, CASES):
    verdict = judge.invoke(JUDGE.format(question=case["question"], reference=case["reference"], answer=row["answer"]))
    row["correct"] = verdict.correct
    row["reason"] = verdict.reason

pd.DataFrame(rows)[["question", "tools_ok", "correct", "reason"]]

,question,tools_ok,correct,reason
0,Vocês têm Duna em estoque?,True,True,"A resposta confirma a disponibilidade do livro 'Duna' e especifica a quantidade, correspondendo exatamente à referência."
1,Quanto custa Neuromancer?,True,True,"A resposta fornece o mesmo preço do livro Neuromancer conforme a referência, sem adicionar informações incorretas."
2,Fundação está disponível?,True,True,"A resposta informa que o livro está esgotado, o que corresponde à informação da referência de que está sem estoque."
3,Têm A Revolução dos Bichos?,True,True,"A resposta confirma que o livro não está disponível no catálogo, alinhando-se com a referência."
4,Qual o horário da loja?,True,False,"A resposta fornece um horário específico da loja, enquanto a referência indica que o atendente não tem essa informação, portanto a resposta não está de acordo com a referência."


O juiz reprovou apenas o caso do horário, pelo motivo certo: a resposta inventa uma informação. A métrica de ferramentas avalia o trace e o juiz avalia a resposta, e cada um detecta erros que o outro não vê.

### Exercício 2

Acrescente a `CASES` um caso com uma pergunta que exija somar os preços de dois livros, com a referência e as ferramentas esperadas, e rode de novo as duas células de avaliação. O agente acertou, e as duas métricas concordaram?

In [15]:
# Seu código aqui

## Custo e regressão

O custo de uma execução é o número de tokens vezes o preço do modelo, cobrado em dólares por milhão de tokens e com preços diferentes para entrada e saída. A função `evaluate` junta a métrica de ferramentas, o juiz, a latência e o custo de cada caso, para avaliar qualquer combinação de modelo e system prompt.

In [16]:
PRICES = {"gpt-4.1-mini": (0.40, 1.60), "gpt-4o-mini": (0.15, 0.60)}


def evaluate(model: str, system: str) -> pd.DataFrame:
    """Roda os casos com o modelo e o system prompt indicados e registra acerto, latência e custo."""
    version = create_agent(
        init_chat_model(f"openai:{model}", temperature=0.0),
        tools=[search_books, get_stock],
        system_prompt=system,
    )
    price_in, price_out = PRICES[model]
    rows = []
    for case in CASES:
        start = time.perf_counter()
        messages = version.invoke({"messages": [{"role": "user", "content": case["question"]}]})["messages"]
        seconds = time.perf_counter() - start
        called, cost = [], 0.0
        for message in messages:
            if message.type == "ai":
                called += [call["name"] for call in message.tool_calls]
                usage = message.usage_metadata
                cost += (usage["input_tokens"] * price_in + usage["output_tokens"] * price_out) / 1e6
        verdict = judge.invoke(JUDGE.format(question=case["question"], reference=case["reference"], answer=messages[-1].text))
        rows.append({
            "question": case["question"],
            "tools_ok": called == case["tools"],
            "correct": verdict.correct,
            "seconds": seconds,
            "cost_usd": cost,
        })
    return pd.DataFrame(rows)

In [17]:
mini = evaluate("gpt-4.1-mini", SYSTEM)
small = evaluate("gpt-4o-mini", SYSTEM)

columns = ["tools_ok", "correct", "seconds", "cost_usd"]
pd.DataFrame({"gpt-4.1-mini": mini[columns].mean(), "gpt-4o-mini": small[columns].mean()}).round(5)

,gpt-4.1-mini,gpt-4o-mini
tools_ok,1.00000,1.00000
correct,0.80000,0.80000
seconds,1.62109,2.43673
cost_usd,0.00020,0.00007


Os dois modelos acertaram os mesmos casos, e o gpt-4o-mini custou cerca de um terço por pergunta. Rodar o mesmo dataset a cada mudança de prompt ou de modelo é o teste de regressão: a versão nova só substitui a anterior se não piorar a tabela. Cinco casos ainda são pouco para decidir a troca, porque os resultados variam entre execuções mesmo com temperatura 0.

### Exercício 3

Acrescente à `SYSTEM` uma frase que obrigue o agente a dizer quando não tiver a informação e avalie a nova versão com o gpt-4.1-mini. O caso do horário passou a ser aprovado pelo juiz, e algum outro caso piorou em relação a `mini`?

In [18]:
# Seu código aqui